# Kaggle Intermediate Machine Learning Course
## This notebook contains solutions, samples and personal testing related to the kaggle course


Score dataset function: Returns the MAE of a Random Forest Regressor

In [42]:
def score_dataset(X_train, X_valid, y_train, y_valid):
    model = RandomForestRegressor(n_estimators=50, random_state=0)
    model.fit(X_train, y_train)

    preds = model.predict(X_valid)
    mae = mean_absolute_error(y_valid, preds)
    return mae

01. Default Kaggle solution:

In [46]:
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split

X_full = pd.read_csv("./data/train.csv", index_col='Id')

features = ['LotArea', 'YearBuilt', '1stFlrSF', '2ndFlrSF', 'FullBath', 'BedroomAbvGr', 'TotRmsAbvGrd']
X = X_full[features].copy()

y = X_full.SalePrice

#split train/validation data:
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.3, random_state=42)

# Define the models
model_1 = RandomForestRegressor(n_estimators=50, random_state=0)
model_2 = RandomForestRegressor(n_estimators=100, random_state=0)
model_3 = RandomForestRegressor(n_estimators=100, criterion='absolute_error', random_state=0)
model_4 = RandomForestRegressor(n_estimators=200, min_samples_split=20, random_state=0)
model_5 = RandomForestRegressor(n_estimators=100, max_depth=7, random_state=0)

models = [model_1, model_2, model_3, model_4, model_5]

from sklearn.metrics import mean_absolute_error

# Function for comparing different models
def score_model(model, X_t=X_train, X_v=X_valid, y_t=y_train, y_v=y_valid):
    model.fit(X_t, y_t)
    preds = model.predict(X_v)
    return mean_absolute_error(y_v, preds)


best_model = models[0]
best_mae = 420000000
for i in range(0, len(models)):
    mae = score_model(models[i])
    print("Model %d MAE: %d" % (i + 1, mae))
    if mae < best_mae:
        best_mae = mae
        best_model = models[i]

print(f"Best model is: {best_model}")

#Work with full data:
X_test_full = pd.read_csv("./data/test.csv", index_col='Id')
X_test = X_test_full[features].copy()

my_model = RandomForestRegressor(n_estimators=100, max_depth=7, random_state=0)
my_model.fit(X, y)

predictions = my_model.predict(X_test)

output = pd.DataFrame({'Id': X_test.index,
                       'SalePrice': predictions})

output.to_csv('./submission.csv', index=False)
print("Output saved at ./submission.csv")

Model 1 MAE: 21713
Model 2 MAE: 21522
Model 3 MAE: 21902
Model 4 MAE: 21561
Model 5 MAE: 21419
Best model is: RandomForestRegressor(max_depth=7, random_state=0)
Output saved at ./submission.csv


02. Missing values section:

In [44]:
from sklearn.metrics import mean_absolute_error

#Read file again:
X_full = pd.read_csv("./data/train.csv", index_col='Id')

# Remove rows with missing target, separate target from predictors
X_full.dropna(axis=0, subset=['SalePrice'], inplace=True)
y = X_full.SalePrice
X_full.drop(['SalePrice'], axis=1, inplace=True)

# To keep things simple, we'll use only numerical predictors
X = X_full.select_dtypes(exclude=['object'])

# Break off validation set from training data
X_train, X_valid, y_train, y_valid = train_test_split(X, y, train_size=0.8, test_size=0.2, random_state=0)

cols_with_missing = [col for col in X_train if X_train[col].isnull().any()]
X_train_dropped = X_train.drop(cols_with_missing, axis=1)
X_valid_dropped = X_valid.drop(cols_with_missing, axis=1)

print("MAE (Drop columns with missing values):")
print(score_dataset(X_train_dropped, X_valid_dropped, y_train, y_valid))

#Now try imputer:
from sklearn.impute import SimpleImputer

imputer = SimpleImputer()
X_train_imputed = pd.DataFrame(imputer.fit_transform(X_train))
X_valid_imputed = pd.DataFrame(imputer.fit_transform(X_valid))

X_train_imputed.columns = X_train.columns
X_valid_imputed.columns = X_valid.columns

print("MAE (Imputation):")
print(score_dataset(X_train_imputed, X_valid_imputed, y_train, y_valid))


MAE (Drop columns with missing values):
17783.55184931507
MAE (Imputation):
17989.222785388134


#### 03.Categorical variables:


In [54]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Read the data
X = pd.read_csv('./data/train.csv', index_col='Id')

# Remove rows with missing target, separate target from predictors
X.dropna(axis=0, subset=['SalePrice'], inplace=True)
y = X.SalePrice
X.drop(['SalePrice'], axis=1, inplace=True)

# To keep things simple, we'll drop columns with missing values
cols_with_missing = [col for col in X.columns if X[col].isnull().any()]
X.drop(cols_with_missing, axis=1, inplace=True)

# Break off validation set from training data
X_train, X_valid, y_train, y_valid = train_test_split(X, y,
                                                      train_size=0.8, test_size=0.2,
                                                      random_state=0)

#First approach - drop columns with categorical values:
drop_X_train =X_train.select_dtypes(exclude = ['object'])
drop_X_valid = X_valid.select_dtypes(exclude = ['object'])

print(f"Score of dropping categorical: {score_dataset(drop_X_train, drop_X_valid, y_train, y_valid)}")

#Second approach - ordinal encoding:

#important - first drop columns that have more values in valid than in test
object_cols = [col for col in X_train if X_train[col].dtype=='object']

good_cols = [col for col in object_cols if set(X_valid[col]).issubset(set(X_train[col]))]
bad_cols = list(set(object_cols) - set(good_cols))

from sklearn.preprocessing import OrdinalEncoder

# Drop categorical columns that will not be encoded
label_X_train = X_train.drop(bad_cols, axis=1)
label_X_valid = X_valid.drop(bad_cols, axis=1)

enc = OrdinalEncoder()
label_X_train[good_cols] =enc.fit_transform(label_X_train[good_cols])
label_X_valid[good_cols] =enc.transform(label_X_valid[good_cols])

print(f"MAE score of ordinal encoding: {score_dataset(label_X_train, label_X_valid, y_train, y_valid)}")

Score of dropping categorical: 17783.55184931507
MAE score of ordinal encoding: 17346.286484018263
